# Aggregate Data and Standard Features Extracted from a Decade of Job Ads

Welcome to the demonstration notebook for the PNAS article: *Aggregate Data and Standard Features Extracted from a Decade of Job Ads*!

The goal of this demonstration is to walk you through the basic usage of all of the JAAT modules, as described in the paper. To do this, we will introduce and develop a working example using USAJOBS data, which we have also open-sourced to enable a reproduction of the analyses we present in the paper.

We have left the outputs pre-filled, but we encourage you to run on your own.

Thank you for checking JAAT out, and here we go!

## Getting Started

First, we will import all of the necessary tools to run JAAT on USAJOBS data. In case you have not already, install all the required dependencies, either via your system or by running the command directly below (uncomment and replace with the correct path).

In [2]:
# !pip install /path/to/requirements.txt

And now for the imports:

In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset

### USAJOBS Data

Let's first get familiar with the USAJOBS dataset we provide. The entire corpus comprise federal job postings from 2017-01 to 2026-03, but for now, we will work on a smaller random sample of 1000 job postings -- this we also provide in our open-source dataset.

In [3]:
sample = load_dataset("loyoladatamining/usajobs", "sample", split="sample")

The above command loads the dataset from Hugging Face, but for ease, we will convert it to a Pandas DataFrame and explore its makeup.

In [4]:
df = sample.to_pandas()
df

,job_id,text,title,company
0,767977200,USAJOBS - Job Announcement family-of-overseas-...,Information Technology Specialist (Policy and ...,None
1,764460600,USAJOBS - Job Announcement family-of-overseas-...,Lead Security Guard,None
2,767905500,USAJOBS - Job Announcement family-of-overseas-...,Lead Recreation Assistant NF-02,None
3,765868200,USAJOBS - Job Announcement family-of-overseas-...,Medical Administrative Specialist,None
4,767266900,USAJOBS - Job Announcement family-of-overseas-...,PROGRAM MANAGER (INFORMATION RESOURCES),None
...,...,...,...,...
995,764122300,USAJOBS - Job Announcement family-of-overseas-...,PROGRAM ANALYST,None
996,764459600,USAJOBS - Job Announcement family-of-overseas-...,CYS Program Associate Technology Lab NF-03,None
997,763842000,USAJOBS - Job Announcement family-of-overseas-...,Licensed Practical Nurse,None
998,767090700,USAJOBS - Job Announcement family-of-overseas-...,SUPERVISORY EDUCATION SERVICES SPECIALIST,None


In [5]:
# see what columns are included
df.columns

Index(['job_id', 'text', 'title', 'company'], dtype='object')

In [6]:
# view a selected job title
df.title.iloc[42]

'Visual Information Specialist (Marketing Assistant)'

In [7]:
# view the above's corresponding job posting text
df.text.iloc[42]

'USAJOBS - Job Announcement family-of-overseas-employees-icon federal-employees-competitive-service-icon federal-employees-competitive-service-icon federal-employees-excepted-service-icon federal-employees-career-transition-icon individuals-with-disabilities-icon Created with Sketch. internal-to-an-agency-icon Created with Sketch. internal-to-an-agency-icon Created with Sketch. military-spouses-icon Created with Sketch. national-guard-and-reserves-icon Created with Sketch. native-americans-icon Created with Sketch. peace-corps-and-americorps-icons Open-to-the-public-icon senior-executive-service-icon se-other students-icon recent-graduates-icon veterans-icon special-hiring-authorities-icon land-and-base-management-icon Visual Information Specialist (Marketing Assistant) Department of the Navy Commander, Navy Installations CNRSE NAVAL AIR STATION JACKSONVILLE N922 Help Summary This position is located in the Morale, Welfare and Recreation (MWR) Marketing Department aboard Naval Air Stat

### JAAT

JAAT, or the *Job Ad Analysis Toolkit* is a modular collection of tools built to extract structured information form job posting text, from task requirements to wage statements to firm names. Below, we will walk through the basic usage of each of the tools we introduce in the paper, demonstrating their ease of use, speed, and structured outputs.

NOTE: we *highly* recommend running this notebook on a machine equipped with a GPU. While JAAT is fully functional on CPU-only setups as well, a GPU will make things go a whole lot quicker!

With that said, let us import the JAAT library. It is as easy as:

In [8]:
import JAAT

In JAAT, we have included some useful utilities to make sure your setup is ready to go.

In [9]:
# run a diagnostic to make sure JAAT recognizes your setup
JAAT.diagnostic()

--- JAAT Diagnostic Report ---
OS: Linux 6.6.87.2-microsoft-standard-WSL2
Python Version: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
PyTorch Version: 2.9.0+cu128
CUDA Available: True
GPU: NVIDIA RTX A6000
VRAM: 51.53 GB
Transformers Version: 4.57.6
------------------------------


In [10]:
# make sure the required files are there
JAAT.setup()

2026-05-14 10:22:30,727 - JAAT - INFO - --- JAAT Setup ---
2026-05-14 10:22:30,732 - JAAT - INFO - Checking JAAT data files...
2026-05-14 10:22:30,739 - JAAT - INFO - ✅ All internal data files present.
2026-05-14 10:22:30,744 - JAAT - INFO - ✅ CUDA detected! Using GPU: NVIDIA RTX A6000
2026-05-14 10:22:30,746 - JAAT - INFO - 
--- Setup Complete ---


In [11]:
# toggle progress bars (if you want!)
JAAT.toggle_progress(True)

2026-05-14 10:22:31,455 - JAAT - INFO - JAAT progress bars are now enabled.


If all looks good with the setup steps, now is the time to dive into the JAAT modules. Let us first pick a representative job posting from our sample dataset.

In [12]:
test_posting = df.iloc[42].text
test_title = df.iloc[42].title
(test_title, test_posting)

('Visual Information Specialist (Marketing Assistant)',
 'USAJOBS - Job Announcement family-of-overseas-employees-icon federal-employees-competitive-service-icon federal-employees-competitive-service-icon federal-employees-excepted-service-icon federal-employees-career-transition-icon individuals-with-disabilities-icon Created with Sketch. internal-to-an-agency-icon Created with Sketch. internal-to-an-agency-icon Created with Sketch. military-spouses-icon Created with Sketch. national-guard-and-reserves-icon Created with Sketch. native-americans-icon Created with Sketch. peace-corps-and-americorps-icons Open-to-the-public-icon senior-executive-service-icon se-other students-icon recent-graduates-icon veterans-icon special-hiring-authorities-icon land-and-base-management-icon Visual Information Specialist (Marketing Assistant) Department of the Navy Commander, Navy Installations CNRSE NAVAL AIR STATION JACKSONVILLE N922 Help Summary This position is located in the Morale, Welfare and Re

Excellent. Now let us load up the first JAAT module: TaskMatch

TaskMatch extract task statements from job postings texts and matches these to standard O*NET codes.

To initialize TaskMatch, run the following (requires a few seconds on GPU to start up):

In [13]:
TM = JAAT.TaskMatch()

2026-05-14 10:22:37,864 - JAAT - INFO - Initalizing TaskMatch...
2026-05-14 10:22:37,866 - JAAT - INFO - Preparing embeddings...


Batches:   0%|          | 0/295 [00:00<?, ?it/s]

2026-05-14 10:22:46,385 - JAAT - INFO - Setting up pipeline...
Device set to use cuda
2026-05-14 10:22:46,680 - JAAT - INFO - Finished.


For most of the JAAT modules, we have built both a single-document mode and a batch mode. We will explore batching later, but first let us see the basic usage of TaskMatch.

In [14]:
TM.get_tasks(test_posting)

[('3242', 'Coordinate with the media to disseminate advertising.'),
 ('1552',
  'Provide information or technical or program assistance to government representatives, employers, or the general public on the issues of public health, environmental protection, or workplace safety.'),
 ('15600',
  'Maintain contact with sponsors to schedule and coordinate site visits or to answer questions about issues such as incomplete data.'),
 ('4000', 'Produce rough and finished graphics and graphic designs.'),
 ('9308',
  'Provide supportive materials for exhibits and displays, such as press kits, advertising, publicity notices, posters, brochures, catalogues, and invitations.'),
 ('7674', 'Develop promotions for current programs and specials.'),
 ('15816', 'Monitor and maintain records of daily facility operations.'),
 ('1783',
  'Oversee publication production, including artwork, layout, computer typesetting, and printing, ensuring adherence to deadlines and budget requirements.'),
 ('3932',
  'Hos

Nice! We can see that the test posting includes many task statements. These have been returned by TaskMatch in a list of tuples, where each tuple takes the form of (TASK ID, TASK DESCRIPTION).

In a similar way, we can load and run SkillMatch, which is similar in makeup to TaskMatch but now identifies and extracts skill statements, coded accordingly to ESCO skill codes.

In [15]:
SM = JAAT.SkillMatch()

2026-05-14 10:22:51,067 - JAAT - INFO - Initializing SkillMatch...
2026-05-14 10:22:51,069 - JAAT - INFO - Preparing embeddings...


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

2026-05-14 10:22:54,497 - JAAT - INFO - Setting up pipeline...
Device set to use cuda
2026-05-14 10:22:54,844 - JAAT - INFO - Finished.


In [16]:
SM.get_skills(test_posting)

[('developing instructive or promotional materials', 'S1.12'),
 ('draw up marketing and sales plan', 'S4.1'),
 ('driving vehicles', 'S8.2'),
 ('lift things', 'T5.1'),
 ('using word processing, publishing and presentation software', 'S5.6'),
 ('developing instructive or promotional materials', 'S1.12'),
 ('digital marketing', 'T1.3'),
 ('prioritise tasks', 'T3.1')]

In addition to extracting information from the job posting text itself, we have also built TitleMatch as a tool to match the job posting's stated job title to O*NET occupation titles.

Running TitleMatch is likewise straightforward:

In [17]:
TiM = JAAT.TitleMatch()

2026-05-14 10:22:59,334 - JAAT - INFO - Initializing TitleMatch...
2026-05-14 10:22:59,335 - JAAT - INFO - Loading data...
2026-05-14 10:22:59,521 - JAAT - INFO - Preparing embeddings...
2026-05-14 10:22:59,522 - JAAT - INFO - Loading title models...
Device set to use cuda
2026-05-14 10:23:01,867 - JAAT - INFO - Finished.


In [18]:
TiM.get_title(test_title)

2026-05-14 10:23:02,957 - JAAT - INFO - Extracting title values...


Processing:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-14 10:23:03,220 - JAAT - INFO - Extracting title features...


Processing:   0%|          | 0/1 [00:00<?, ?it/s]

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
2026-05-14 10:23:03,493 - JAAT - INFO - Matching titles to codes...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[('25-4022.00', 0.921, -4.5, 'none')]

The returns a tuple of information: (TITLE CODE, MATCH SCORE, TITLE_VALUE, TITLE_FEATURES)

25-4022.00 = "Librarians and Media Collections Specialists"


The title "value" is the result of a model who assigns a numeric score to a title based on its seniority or experience level, ranging from -10 to 60. Title "features" are interesting attributes of the job title, if any, such as full-time vs part-time.

While the tools so far have been useful for matching information contained within job postings to standard coded taxonomies, we also may wish to extract information directly from the text itself. For this, we have built two tools for firm name extraction and wage data extraction Let's first look at firms..

In [19]:
FE = JAAT.FirmExtract()

Device set to use cuda


In [20]:
FE.get_firm(test_posting)

For ChunkPipeline using num_workers>0 is likely to result in errors since everything is iterable, setting `num_workers=1` to guarantee correctness.
/home/sjmeis/.local/lib/python3.10/site-packages/transformers/pipelines/token_classification.py:490: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


('NASJACKSONVILLE', 0.6029296517372131)

This matches well with "NAVAL AIR STATION JACKSONVILLE".

Let's also see if we can extract any wage information out of the job posting text.

In [21]:
WE = JAAT.WageExtract()

Device set to use cuda
Device set to use cuda
Device set to use cuda


In [22]:
WE.get_wage(test_posting)

For ChunkPipeline using num_workers>0 is likely to result in errors since everything is iterable, setting `num_workers=1` to guarantee correctness.


{'min': '18', 'max': '22', 'frequency': 'hourly'}

This checks out as well! These tools become even more useful when processing data at scale, and especially when structured metaata (e.g., firm or wage) is not readily available.

We introduce below one final tool, which is in beta and was introduced in the paper: AIMatch. With the rising prominence of AI in the workplace, it becomes very important to be able to map and extract the AI-related tasks and skills that may be required on the job. We built AIMatch to do this!

In [23]:
AI = JAAT.AIMatch()

2026-05-14 10:23:17,171 - JAAT - INFO - Initializing AIMatch...
2026-05-14 10:23:17,174 - JAAT - INFO - Preparing embeddings...


Batches:   0%|          | 0/308 [00:00<?, ?it/s]

2026-05-14 10:23:24,563 - JAAT - INFO - Setting up pipeline...
Device set to use cuda
2026-05-14 10:23:24,930 - JAAT - INFO - Finished.


In [24]:
AI.get_ai(test_posting)

([('Provides technical assistance across project management, policy, communication, research support, service development, training, and advisory roles.',
   'A6.G.6.k.14020'),
  ('Skilled in creating publishable content, infographics, and actionable deliverables, including CTRC orders.',
   'A6.F.10.i.13211'),
  ('Participates in and leads technical committees, meetings, and workshops for knowledge sharing and collaboration.',
   'A6.H.6.b.14717'),
  ('Proficient in office suite applications and various software packages with strong overall computer skills.',
   'A6.B.1.j.13939'),
  ('Ability to instill and employ best practices in software development and documentation, ensuring designs meet requirements.',
   'A6.F.0.n.16322'),
  ('Demonstrates a strong passion for technology and cybersecurity, with a commitment to continuous learning and teaching others.',
   'A6.J.0.k.15669'),
  ('Ability to prepare technical documentation, reports, and presentation materials, including briefings 

The output of AIMatch requires some explanantion. In our building of the tool, we adopted a broad definition of "AI" and accordingly, built an "AI score" to help users quantify how "AI" the resulting outputs are.

The first part of the outputs, the tuples, stem from a custom-built taxonomy of AI-related tasks and skills (work in progress). Each of these matches is associated to an extraction score:

[0.828, 0.507, 0.748, 0.584, 0.601, 0.655, 0.565, 0.552, 0.556]

and a match score:

[0.88, 0.88, 0.879, 0.918, 0.898, 0.898, 0.884, 0.892, 0.88]

As you can see, some of the extraction scores are low, and we provide the user the flexibility to set thresholds or discard results as necessary. The match scores should be interpreted relatively. With our default model: Anything above 0.90 (out of 1) is a good match, as as we descend deeper into the 0.8 territory, the matches should be treated with less confidence.

As a final piece to the output, you can see that the 9 matches contribute to an average AI score of 0.697 - moderately high but not "frontier" work. We note that these measures are still in development and will be refined.

### Wrap-up

That was the end of the basic introduction of JAAT!

We will now move on to more advanced usage, namely on larger amounts of data, enabling you to replicate our work on USAJOBS.

## Scaling Up

Mow that we have introduced JAAT and its many functions, we will now demonstrate how to scale up its usage. The end goal of this process is the build structured, coded data files from large corpora of unstructured texts. These codes files are the foundation of the analyses we provide in the paper, and they are the potential future foundation for any ideas you might have!

For now, we will remain with the sample of 1000 USAJOBS postings. Later on, we provide the code to iterate through all of the USAJOBS monthly data, such that the true collection of monthly coded files can be constructed.

Here's a reminder of what the sample looks like:

In [25]:
df

,job_id,text,title,company
0,767977200,USAJOBS - Job Announcement family-of-overseas-...,Information Technology Specialist (Policy and ...,None
1,764460600,USAJOBS - Job Announcement family-of-overseas-...,Lead Security Guard,None
2,767905500,USAJOBS - Job Announcement family-of-overseas-...,Lead Recreation Assistant NF-02,None
3,765868200,USAJOBS - Job Announcement family-of-overseas-...,Medical Administrative Specialist,None
4,767266900,USAJOBS - Job Announcement family-of-overseas-...,PROGRAM MANAGER (INFORMATION RESOURCES),None
...,...,...,...,...
995,764122300,USAJOBS - Job Announcement family-of-overseas-...,PROGRAM ANALYST,None
996,764459600,USAJOBS - Job Announcement family-of-overseas-...,CYS Program Associate Technology Lab NF-03,None
997,763842000,USAJOBS - Job Announcement family-of-overseas-...,Licensed Practical Nurse,None
998,767090700,USAJOBS - Job Announcement family-of-overseas-...,SUPERVISORY EDUCATION SERVICES SPECIALIST,None


As mentioned earlier, we have designed all of the JAAT tools for the primary use case of *batch processing* -- running JAAT on thousands to millions of job postings. To introduce this capability, first a few helpful utilities.

The first is text batching tool. In the case of our 1000 texts, it might not make too much sense, but if your corpus is much larger, you may want to process it in batches. An example of its usage is below.

In [26]:
# break down our data into chunks of 100 texts each
batches1 = JAAT.chunker(df.text.to_list(), size=100)
print(batches1)
print(len(list(batches1)))

# or, break down our data into chunks of any chosen size, say 42
batches2 = JAAT.chunker(df.text.to_list(), size=42)
print(batches2)
print(len(list(batches2)))

# what is in the return?
batches2 = JAAT.chunker(df.text.to_list(), size=42)
print("---------")
for i, batch in enumerate(batches2):
    print("Batch {}: {} postings".format(i, len(batch)), end = ', ')

<generator object chunker at 0x74b7de8f0cf0>
10
<generator object chunker at 0x74b7de8f3ae0>
24
---------
Batch 0: 42 postings, Batch 1: 42 postings, Batch 2: 42 postings, Batch 3: 42 postings, Batch 4: 42 postings, Batch 5: 42 postings, Batch 6: 42 postings, Batch 7: 42 postings, Batch 8: 42 postings, Batch 9: 42 postings, Batch 10: 42 postings, Batch 11: 42 postings, Batch 12: 42 postings, Batch 13: 42 postings, Batch 14: 42 postings, Batch 15: 42 postings, Batch 16: 42 postings, Batch 17: 42 postings, Batch 18: 42 postings, Batch 19: 42 postings, Batch 20: 42 postings, Batch 21: 42 postings, Batch 22: 42 postings, Batch 23: 34 postings, 

The `chunker` utility breaks down potentially large collections of texts into a *generator* object, which is essentially a queue of the batches of texts. You only need to handle one at a time!

In addition, you can also use the following utility:

In [27]:
test_batch = list(JAAT.chunker(df.title.to_list(), size=10))[0] # take the first batch
JAAT.validate_inputs(test_batch)

['Information Technology Specialist (Policy and Planning)',
 'Lead Security Guard',
 'Lead Recreation Assistant NF-02',
 'Medical Administrative Specialist',
 'PROGRAM MANAGER (INFORMATION RESOURCES)',
 'Supervisory Interdisciplinary Engineer/Architect',
 'Registered Nurse - Health Promotion Disease Prevention Program Manager',
 'Instructional Systems Specialist (Analyst), GS-1750-12',
 'Health Technician (Telehealth Clinical)',
 'Psychology Technician']

This utility is helpful in validating that all elements of a batch (or any list) is valid for input to JAAT, i.e., not null or missing. This helps avoid errors down the road!

Now we can put all these tools together, and showcase batch functionality. Let's use TaskMatch!

In [28]:
batches = JAAT.chunker(df.text.to_list(), size=100) # smaller batches of 100 postings each

results = []
for i, batch in enumerate(batches):
    print("Processing batch {}...".format(i))
    validated_batch = JAAT.validate_inputs(batch) # comment out for quicker processing
    res = TM.get_tasks_batch(validated_batch)
    results.extend(res)

Processing batch 0...
Processing batch 1...
Processing batch 2...
Processing batch 3...
Processing batch 4...
Processing batch 5...
Processing batch 6...
Processing batch 7...
Processing batch 8...
Processing batch 9...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Congratulations, you just processed 1000 job postings through TaskMatch!

As you can see, we captured the batch outputs of TaskMatch into `results`.

In [29]:
len(results)

1000

Each of the elements in this list corresponds 1:1 to the input job postings from our USAJOBS sample. For example:

In [30]:
df.iloc[123].text

'USAJOBS - Job Announcement family-of-overseas-employees-icon federal-employees-competitive-service-icon federal-employees-competitive-service-icon federal-employees-excepted-service-icon federal-employees-career-transition-icon individuals-with-disabilities-icon Created with Sketch. internal-to-an-agency-icon Created with Sketch. internal-to-an-agency-icon Created with Sketch. military-spouses-icon Created with Sketch. national-guard-and-reserves-icon Created with Sketch. native-americans-icon Created with Sketch. peace-corps-and-americorps-icons Open-to-the-public-icon senior-executive-service-icon se-other students-icon recent-graduates-icon veterans-icon special-hiring-authorities-icon land-and-base-management-icon Health Technician (Telehealth Clinical) (Advanced) Department of Veterans Affairs Veterans Health Administration Veterans Health Administration Help Summary The Health Technician (Telehealth Clinical) Advanced provides high level support, training, and advanced managemen

In [31]:
results[123]

[('18447',
  'Develop and implement individualized plans for health care management.'),
 ('529', 'Set up equipment and prepare medical treatment rooms.'),
 ('22814', 'Operate digital picture archiving communications systems.'),
 ('4266',
  'Assemble, clean, or maintain equipment or materials for patient use.'),
 ('18389',
  'Develop treatment plans, based on scientific rationale, standards of care, and professional practice guidelines.'),
 ('18097',
  'Develop or implement policies or practices to ensure the privacy, confidentiality, or security of patient information.'),
 ('5074', 'Answer telephones and schedule appointments.'),
 ('1818',
  'Refer patients to other health professionals or agencies when appropriate.'),
 ('18091',
  'Plan, install, repair, or troubleshoot telehealth technology applications or systems in homes.'),
 ('20417',
  'Design patient education programs that include information required to make informed health care and treatment decisions.'),
 ('5435', 'Measure a

With this structure, we can easily and quickly "wrap up" results and map them back to the original posting IDs:

In [32]:
codes = [";".join([c[0] for c in x]) for x in results]
result_df = pd.DataFrame()
result_df["job_id"] = df["job_id"]
result_df["tasks"] = codes
result_df

,job_id,tasks
0,767977200,16156;16270;18595;2785;14623;9719;22;21702;213...
1,764460600,22942;23159;22011;8480;2703;2775;2133;9809;183...
2,767905500,4480;2671;21987;2791;5258;7236;1082;17521
3,765868200,4268;21876;14339;18871
4,767266900,8469;16169;7663;14758;3481;21778;21496;15677;1...
...,...,...
995,764122300,3751;1554;15649;7379;7572;1506;14680;9007;7205...
996,764459600,1914;21408;22497;4565;20130;844;6480;663;1326;...
997,763842000,18308;17286;22864;17094;19015;17326;2637;17199...
998,767090700,100;1862;23169;18595;3395;15258;15652;6979;567...


And voila, you have now gone from job postings to structured, coded data using JAAT. A similar processing can be followed for all JAAT modules, you can test them out below!

In [33]:
all_texts = df.text.to_list()
skills = SM.get_skills_batch(all_texts)
codes = [";".join([c[1] for c in x]) for x in skills]
result_df["skills"] = codes

In [34]:
titles = TiM.get_title(df.title.to_list())
codes = [x[0] for x in titles]
result_df["title"] = codes

2026-05-14 10:26:10,750 - JAAT - INFO - Extracting title values...


Processing:   0%|          | 0/1000 [00:00<?, ?it/s]

2026-05-14 10:26:13,343 - JAAT - INFO - Extracting title features...


Processing:   0%|          | 0/63 [00:00<?, ?it/s]

2026-05-14 10:26:15,629 - JAAT - INFO - Matching titles to codes...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [37]:
firms = FE.get_firm_batch(all_texts)
temp = [x[0] for x in firms]
result_df["firm"] = temp

In [39]:
wages = WE.get_wage_batch(all_texts)
temp = ["{}-{}-{}".format(str(x["min"]), str(x["max"]), str(x["frequency"])) if x is not None else None for x in wages]
result_df["wage"] = temp

In [41]:
ai = AI.get_ai_batch(all_texts)
codes = [";".join([c[1] for c in x]) for x in ai[0]]
result_df["AI"] = codes

And with that, we now have a complete coded dataset with the JAAT-provided features, mapped to each of the original 1000 job postings from the USAJOBS sample.

In [42]:
result_df

,job_id,tasks,skills,title,firm,wage,AI
0,767977200,16156;16270;18595;2785;14623;9719;22;21702;213...,S2.3;S2.9;S1.9;T3.1;S1.5;S3.1;S3.1;S1.5;S4.7;S...,15-1232.00,None,"76,990-100,084-annually",A6.G.5.n.12305;A6.A.15.k.16790;A6.I.1.c.18983;...
1,764460600,22942;23159;22011;8480;2703;2775;2133;9809;183...,T3.1;S8.7;S3.4;S3.3;S1.8,33-9032.00,None,"46,696-60,703-annually",A6.A.4.b.12431
2,767905500,4480;2671;21987;2791;5258;7236;1082;17521,S3.4;S1.1;S8.0;S1.2;S1.14;S1.8,39-3091.00,None,17-19-hourly,
3,765868200,4268;21876;14339;18871,T4.2;T5.2;S6.2;S1.7,29-2072.00,None,"69,107-89,835-annually",A6.C.2.g.13502;A6.G.6.k.11216;A6.A.4.d.13737
4,767266900,8469;16169;7663;14758;3481;21778;21496;15677;1...,S4.4;T2.2;T2.3;S2.7;S2.8;S2.7;S4.1,15-1299.09,None,"112,015-145,617-annually",A6.A.13.f.11619;A6.F.8.d.11570;A6.G.2.j.21158;...
...,...,...,...,...,...,...,...
995,764122300,3751;1554;15649;7379;7572;1506;14680;9007;7205...,S1.5;S1.5;S1.0;T3.1;T3.1;S4.2;S4.2;S4.2;T4.2;S...,13-1111.00,TRANSPORTATION SECURITY,"71,099-109,908-annually",A6.D.2.d.13930;A6.B.0.d.14515;A6.J.9.b.23286;A...
996,764459600,1914;21408;22497;4565;20130;844;6480;663;1326;...,S4.8;T2.2;S4.8;T1.3;T1.3;T1.3;S6.2,19-4099.01,None,"37,696-49,009-annually",A6.A.6.h.10945;A6.G.4.g.17559;A6.B.1.l.11427;A...
997,763842000,18308;17286;22864;17094;19015;17326;2637;17199...,S2.2;S3.2;S3.1;T3.2;T4.1;S3.2;S2.8;S1.3,29-2061.00,VETERANS HEALTH ADMIN,"60,958-79,241-annually",
998,767090700,100;1862;23169;18595;3395;15258;15652;6979;567...,S4.8;S4.1;T3.2;T2.4;S4.8;S2.7;S4.1;S4.3;T6.6;T4.2,11-9032.00,None,"69,107-89,835-annually",A6.F.6.c.11235;A6.F.6.c.11235;A6.B.2.z.20348;A...


## Running JAAT on the USAJOBS Corpus

Moving beyond the 1000 posting sample with which we have been working, we are now equipped to run JAAT on all of the USAJOBS data. This is simplified by the ability stream all of the USAJOBS data from our open-source Hugging Face repository.

We will walk you through the basic steps below. Please note that depending on your setup and hardware constraints, you may need to modify the pipeline slightly. In addition, some of the processing below may take very long (USAJOBS is over three million postings), so plan accordingly! We also highly recommend moving some of the logic to shell scripts -- this Juypter Notebook is good for demonstration, but not necessarily for large-scale processing!

In [52]:
from datasets import get_dataset_split_names
from tqdm.auto import tqdm

First let's explore a little of the makeup of our USAJOBS repository.

In [47]:
splits = get_dataset_split_names("loyoladatamining/usajobs", "postings")
print(splits)

['2026_03', '2026_02', '2026_01', '2025_12', '2025_11', '2025_10', '2025_09', '2025_08', '2025_07', '2025_06', '2025_05', '2025_04', '2025_03', '2025_02', '2025_01', '2024_12', '2024_11', '2024_10', '2024_09', '2024_08', '2024_07', '2024_06', '2024_05', '2024_04', '2024_03', '2024_02', '2024_01', '2023_12', '2023_11', '2023_10', '2023_09', '2023_08', '2023_07', '2023_06', '2023_05', '2023_04', '2023_03', '2023_02', '2023_01', '2022_12', '2022_11', '2022_10', '2022_09', '2022_08', '2022_07', '2022_06', '2022_05', '2022_04', '2022_03', '2022_02', '2022_01', '2021_12', '2021_11', '2021_10', '2021_09', '2021_08', '2021_07', '2021_06', '2021_05', '2021_04', '2021_03', '2021_02', '2021_01', '2020_12', '2020_11', '2020_10', '2020_09', '2020_08', '2020_07', '2020_06', '2020_05', '2020_04', '2020_03', '2020_02', '2020_01', '2019_12', '2019_11', '2019_10', '2019_09', '2019_08', '2019_07', '2019_06', '2019_05', '2019_04', '2019_03', '2019_02', '2019_01', '2018_12', '2018_11', '2018_10', '2018_09'

As you can see, the dataset repository is organized into a number of "splits", or partitions, of the larger corpus. Naturally, this is done by month.

We can use the *stream* feature of the dataset loader to avoid having to store the entire dataset at once in memory. Note that this requires an active internet connection.

In [50]:
sample_stream = load_dataset("loyoladatamining/usajobs", "postings", split="2017_10", streaming=True)

In [51]:
# iterate through a few examples from the dataset stream
idx = 0
for posting in sample_stream:
    if idx == 10:
        break
    print(posting["job_id"])
    idx += 1

382487500
453902600
453256800
453669700
455916400
463982300
453924400
452693700
462547700
458246500


With this setup at our disposal, we can now set up a "pipeline" to iterate through all of the splits, stream the data as needed, and then process this data through our JAAT modules. Here is an example with TaskMatch (this will run for a while, so by default, we break after the first split is processed).

In [53]:
master_results = {}
for s in splits:
    print(s)
    stream = load_dataset("loyoladatamining/usajobs", "postings", split=s, streaming=True)

    # temporary data structures to hold the streamed data
    job_ids = []
    job_postings = []
    job_titles = []
    for posting in tqdm(stream):
        job_ids.append(posting["job_id"])
        job_postings.append(posting["text"])
        job_titles.append(posting["title"])

    monthly_df = pd.DataFrame()
    monthly_df["job_id"] = job_ids
    
    # now process in batch mode
    results = TM.get_tasks_batch(job_postings)
    codes = [";".join([c[0] for c in x]) for x in results]
    monthly_df["tasks"] = codes

    titles = TiM.get_title(job_titles)
    codes = [x[0] for x in titles]
    monthly_df["title"] = codes

    # ...
    # and likewise for other JAAT modules

    master_results[s] = monthly_df

    # manual cleanup - optional, but good to do
    del job_ids[:]
    del job_postings[:]
    del job_titles[:]

    # uncoment for full run!
    break

2026_03


0it [00:00, ?it/s]

2026-05-14 11:04:25,787 - JAAT - INFO - Extracting title values...


Processing:   0%|          | 0/14145 [00:00<?, ?it/s]

2026-05-14 11:04:54,721 - JAAT - INFO - Extracting title features...


Processing:   0%|          | 0/885 [00:00<?, ?it/s]

2026-05-14 11:05:23,532 - JAAT - INFO - Matching titles to codes...


Batches:   0%|          | 0/443 [00:00<?, ?it/s]

With this finished, you will have a data structure that maps each month to its respective results DataFrame:

In [59]:
master_results

{'2026_03':           job_id                                              tasks  \
 0      816513500  5177;5177;9860;16519;18788;5177;20812;1452;299...   
 1      833603300                                          2127;8568   
 2      833725700  2702;1649;2702;1649;21647;18578;2702;3363;2388...   
 3      833226500  2702;1649;2702;1649;21647;18578;2702;3363;2388...   
 4      834963900  14304;15453;19920;15452;14574;22276;22088;1475...   
 ...          ...                                                ...   
 14140  860094700   4543;657;20437;8149;5269;14457;12895;16977;16977   
 14141  860953100  20432;17;20432;17;2402;11178;2052;6998;8872;25...   
 14142  860875700  23583;21193;23583;21193;23168;5049;23457;22987...   
 14143  859781300  9658;23121;9491;8053;11192;11198;22261;5053;95...   
 14144  860084700  23118;18856;983;14632;17603;8419;8568;8830;857...   
 
             title  
 0      49-3042.00  
 1      15-1221.00  
 2      13-1141.00  
 3      13-1141.00  
 4      51-8093.00

Now with all of the saved results, we save the DataFrames to flat files, for use in downstream ETL and analysis piplines.

In [56]:
#for month in master_results:
#    master_results[month].to_csv("/path/to/output/{}.csv".format(month), index=False) # change path to your desired location!

### Wrap up

This concludes the full demonstration on how to use JAAT, from single uses to full scale data processing pipelines on large text corpora. Follow along our other demo notebooks for a snapshot and replication code of how we evaluated JAAT with the help of modern LLMs, as well as some interesting use cases of JAAT.